# 🔢 Calenda — 양자화 전용 노트북 (Q8_0 배포)

학습 노트북과 **분리**된 독립 양자화 파이프라인. 매 학습마다 돌릴 필요 없이, 배포할 라운드가 정해졌을 때만 실행한다.

**모델 소스 3가지** (셀 2의 `SOURCE`):
- `upload` — 학습 노트북이 받은 `lora_{NAME}.zip`(어댑터 ~50MB) 업로드 → 자동 merge → 양자화 (권장)
- `local` — 로컬 실행 시 디스크의 `models/merged/` 또는 `models/lora/`를 그대로 사용
- `git` — 어댑터를 `git add -f`로 커밋·푸시한 경우 repo clone으로 수신

> ⚠️ `.gitignore`가 `models/**`·`*.safetensors`·`*.gguf`를 제외하므로, **그냥 push로는 모델이 git에 안 올라간다.** `git` 소스를 쓰려면 학습 노트북에서 `!git add -f models/lora/{NAME}` 후 커밋·푸시할 것(어댑터만; merged 1.2GB는 비권장). 기본은 `upload`.

## 1. repo clone + 설정 (configs·scripts 필요)

In [ ]:
import os
%cd /content
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /content/Calenda
import yaml, os
CONFIG = 'configs/train_qwen3_0_6b.yaml'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE       = _mcfg['hf_id']
NAME       = os.path.basename(_cfg['output_dir'])
LORA_DIR   = _cfg['output_dir']
MERGED_DIR = f'models/merged/{NAME}'
GOLDEN     = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
print('NAME', NAME, '| BASE', BASE)
print('LORA_DIR', LORA_DIR, '| MERGED_DIR', MERGED_DIR)

## 2. 모델 소스 선택 + merged 확보
`SOURCE`를 골라 실행. merged가 없고 어댑터만 있으면 자동 merge.

In [ ]:
SOURCE = "upload"   # "upload" | "local" | "git"

import os, glob, zipfile
!pip install -q -e .[train]

def have_merged():
    return os.path.isdir(MERGED_DIR) and glob.glob(MERGED_DIR + '/*.safetensors')
def have_lora():
    return os.path.isdir(LORA_DIR) and glob.glob(LORA_DIR + '/adapter_*')

if not have_merged():
    if not have_lora():
        if SOURCE == 'upload':
            from google.colab import files
            print(f'>> lora_{NAME}.zip (어댑터) 를 업로드하세요')
            up = files.upload()
            zf = [f for f in up if f.endswith('.zip')]
            assert zf, '.zip 업로드 필요'
            os.makedirs(LORA_DIR, exist_ok=True)
            zipfile.ZipFile(zf[0]).extractall(LORA_DIR)
            print('어댑터 압축해제 →', LORA_DIR)
        elif SOURCE == 'git':
            raise SystemExit(f'git에 어댑터 없음. 학습 노트북에서 "!git add -f {LORA_DIR} && git commit && git push" 후 다시 시도.')
        else:  # local
            raise SystemExit(f'local 모드인데 {MERGED_DIR} 도 {LORA_DIR} 도 없음.')
    print('[merge] LoRA → FP16')
    !python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
else:
    print('merged 이미 존재 → merge 스킵')
assert have_merged(), 'merged 확보 실패'
print('MERGED_DIR 준비 완료:', MERGED_DIR)

## 3. 양자화 (merged → Q8_0 배포 · Q4_K_M 비교). CPU 작업. llama.cpp 빌드 처음 1회 ~5분.

In [ ]:
import os, subprocess
LLAMA = '/content/llama.cpp'
GGUF_DIR = 'models/gguf/' + NAME
os.makedirs(GGUF_DIR, exist_ok=True)
QUANT = LLAMA + '/build/bin/llama-quantize'
def run(cmd, log):
    rc = subprocess.run(cmd, shell=True, stdout=open(log,'w'), stderr=subprocess.STDOUT).returncode
    if rc: print('  ! fail rc=', rc, open(log).read()[-1500:])
    return rc
def mb(x): return str(round(os.path.getsize(x)/1e6)) + 'MB'
print('[1/4] llama.cpp')
if not os.path.isdir(LLAMA): run('git clone --depth 1 https://github.com/ggml-org/llama.cpp.git ' + LLAMA, '/tmp/clone.log')
run('pip install -q gguf', '/tmp/pip.log')
if not os.path.isfile(QUANT):
    print('[2/4] cmake build (~5min)')
    run('cmake -S ' + LLAMA + ' -B ' + LLAMA + '/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF', '/tmp/cfg.log')
    run('cmake --build ' + LLAMA + '/build -j --target llama-quantize', '/tmp/build.log')
    assert os.path.isfile(QUANT), open('/tmp/build.log').read()[-2000:]
    print('     done')
else:
    print('[2/4] build skip')
print('[3/4] f16 convert')
F16 = GGUF_DIR + '/' + NAME + '.f16.gguf'
run('python ' + LLAMA + '/convert_hf_to_gguf.py ' + MERGED_DIR + ' --outfile ' + F16 + ' --outtype f16', '/tmp/convert.log')
print('     f16', mb(F16))
print('[4/4] quantize')
for q in ['Q8_0', 'Q4_K_M']:
    out = GGUF_DIR + '/' + NAME + '.' + q + '.gguf'
    if run(QUANT + ' ' + F16 + ' ' + out + ' ' + q, '/tmp/q_' + q + '.log') == 0: print('    ', q, mb(out))
print('done', NAME, '->', GGUF_DIR)

## 4. (선택) GGUF 골든 검증 — Q8_0 vs Q4_K_M. 양자화 후 실품질 확인. CPU 추론.

In [ ]:
!pip install -q llama-cpp-python
import json, os
GGUF_DIR = f'models/gguf/{NAME}'
res = {}
for q in ['Q8_0', 'Q4_K_M']:
    outj = f'logs/eval_{NAME}.{q}.json'
    !python scripts/eval_gguf.py --gguf {GGUF_DIR}/{NAME}.{q}.gguf --eval {GOLDEN} --out {outj} --failures_out data/failures/{NAME}.{q}.fail.jsonl --n_gpu_layers 0
    res[q] = json.load(open(outj, encoding='utf-8'))
KEYS = ['final_score','json_valid_rate','title_f1_avg','time_match_rate','location_f1_avg','schedule_status_acc','event_count_acc']
print()
print(f"{'metric':<20}{'Q8_0':>9}{'Q4_K_M':>9}")
for k in KEYS:
    print(f"{k:<20}{res['Q8_0'].get(k,0):>9.3f}{res['Q4_K_M'].get(k,0):>9.3f}")

## 5. GGUF 다운로드 (배포본 **Q8_0**)

In [ ]:
import os
from google.colab import files
GGUF_DIR = f'models/gguf/{NAME}'
for q in ['Q8_0', 'Q4_K_M']:
    p = f'{GGUF_DIR}/{NAME}.{q}.gguf'
    if os.path.isfile(p):
        print('download:', p, round(os.path.getsize(p)/1e6), 'MB')
        files.download(p)